# YOLOv8n 학습 · 평가 · RKNN용 ONNX 만들기

**바꿀 곳은 아래 "① 설정" 셀 하나뿐이다.** 설정을 고친 뒤 메뉴 **런타임 → 모두 실행**.

| 단계 | 결과 (Drive에 저장) |
|---|---|
| 학습 | `best.pt` |
| Test 세트 평가 | `metrics.json` (mAP50 등 → `docs/METRICS.md`에 기록) |
| RKNN용 ONNX export | `best_rknn.onnx` (출력 9개, 640×640) |
| 캘리브레이션 목록 | `dataset.txt` + `calib/` (Train 이미지만) |

다음 단계: 같은 Drive 폴더의 파일로 `convert_rknn.ipynb`를 실행해 `.rknn`을 만든다.

준비:
1. 런타임 → 런타임 유형 변경 → **GPU (T4)**
2. Roboflow에서 데이터셋을 **YOLOv8 형식 zip**으로 내보내 Drive의 `MyDrive/jarvis/data/vision/`에 올린다
   - Roboflow에서 **train / valid / test** 세트가 모두 있어야 한다. 세트는 **촬영 세션 단위**로 나눈다 (같은 세션 사진이 train과 test에 섞이면 안 된다)

절차·근거: `recipes/yolo-to-rknn.md` · 문제가 생기면 출력 전체를 팀 메신저나 이슈에 올린다.

In [ ]:
# ① 설정 — 여기만 바꾼다
DATA_ZIP = "/content/drive/MyDrive/jarvis/data/vision/kitchen_v1.zip"  # Roboflow YOLOv8 zip
RUN_NAME = "kitchen_v1"  # 결과 폴더 이름 (실험마다 다르게: kitchen_v1, kitchen_v2 ...)
EPOCHS = 100
IMGSZ = 640
BATCH = 16
CALIB_N = 200  # 캘리브레이션에 쓸 Train 이미지 수
DRIVE_OUT = "/content/drive/MyDrive/jarvis/models/vision"

In [ ]:
# ② Drive 연결 · GPU 확인
from google.colab import drive

drive.mount("/content/drive")
!nvidia-smi -L || echo "⚠️ GPU 없음: 런타임 → 런타임 유형 변경 → GPU"

In [ ]:
# ③ 코드·학습 도구 설치 (버전 고정: docs/conventions/versions.md)
!git clone -q https://github.com/samsung-team-jarvis/jarvis.git /content/jarvis || (cd /content/jarvis && git pull -q)
!pip install -q "ultralytics==8.4.171"

In [ ]:
# ④ 데이터 풀기 · data.yaml 경로 정리 (Roboflow zip의 상대 경로를 이 환경 기준으로 다시 씀)
import pathlib
import zipfile

import yaml

data_dir = pathlib.Path("/content/data") / RUN_NAME
data_dir.mkdir(parents=True, exist_ok=True)
zipfile.ZipFile(DATA_ZIP).extractall(data_dir)

src_yaml = next(data_dir.rglob("data.yaml"))
root = src_yaml.parent
cfg = yaml.safe_load(src_yaml.read_text())
splits = {"train": "train/images", "val": "valid/images", "test": "test/images"}
fixed = {"path": str(root), "names": cfg["names"]}
for key, rel in splits.items():
    if (root / rel).exists():
        fixed[key] = rel
assert "train" in fixed and "val" in fixed, (
    "train / valid 폴더가 없습니다 (Roboflow에서 YOLOv8 형식인지 확인)"
)
if "test" not in fixed:
    print("⚠️ test 세트가 없어 valid로 평가합니다. Roboflow에서 test 세트를 만드는 것을 권장")
DATA_YAML = root / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(fixed, allow_unicode=True))
counts = {k: len(list((root / v).glob("*"))) for k, v in fixed.items() if k in splits}
print("클래스:", cfg["names"])
print("이미지 수:", counts)

In [ ]:
# ⑤ 학습 → Test 평가 (metrics.json)
OUT = f"/content/runs/{RUN_NAME}"
!python /content/jarvis/training/yolo/train_eval.py --data {DATA_YAML} --epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} --device 0 --out {OUT}

In [ ]:
# ⑥ export 전용 환경 (학습 환경과 버전이 달라 분리: numpy<2, torch 2.4.1)
!pip install -q uv
!uv venv -q /content/export-env --python 3.11
!uv pip install -q --python /content/export-env/bin/python -r /content/jarvis/training/yolo/export-requirements.txt

In [ ]:
# ⑦ RKNN용 ONNX export (airockchip 포크)
!PYTHON=/content/export-env/bin/python bash /content/jarvis/training/yolo/export_rknn_onnx.sh {OUT}/weights/best.pt {OUT}/best_rknn.onnx

In [ ]:
# ⑧ 캘리브레이션 목록 (Train 이미지만, 다른 런타임에서도 쓰도록 이미지 복사)
!python /content/jarvis/training/yolo/make_calib_list.py --images {root}/train/images --n {CALIB_N} --out {OUT}/dataset.txt --copy

In [ ]:
# ⑨ Drive에 저장
import json
import shutil

dest = pathlib.Path(DRIVE_OUT) / RUN_NAME
dest.mkdir(parents=True, exist_ok=True)
for name in ["weights/best.pt", "metrics.json", "best_rknn.onnx", "dataset.txt"]:
    shutil.copy2(f"{OUT}/{name}", dest / pathlib.Path(name).name)
shutil.copytree(f"{OUT}/calib", dest / "calib", dirs_exist_ok=True)
shutil.copy2(DATA_YAML, dest / "data_colab.yaml")
print("저장:", dest)
print(json.dumps(json.load(open(f"{OUT}/metrics.json")), ensure_ascii=False, indent=2))

## 끝나면

1. 위 metrics의 `map50`, `map50_95`, `precision`, `recall`을 `docs/METRICS.md` 측정 로그에 기록한다
   - 측정 위치: **Colab (학습 환경, fp32)** — 보드 수치가 아니다
   - 데이터셋 이름, 이미지 수, RUN_NAME, 날짜를 같이 적는다
2. `convert_rknn.ipynb`를 **새 런타임**에서 열고, 우리 모델 섹션의 경로를 이 폴더로 바꿔 `.rknn`을 만든다
   - `--onnx {DRIVE_OUT}/{RUN_NAME}/best_rknn.onnx`
   - `--dataset {DRIVE_OUT}/{RUN_NAME}/dataset.txt`